## 2. Reward Model

In this section, we train a **reward model** to evaluate the quality or relevance of generated content.  
We skip the earlier stages of the LLM pipeline — **(1) pretraining** and **(2) supervised fine-tuning** — and start from an already **instruction-tuned model**.

---

### Objective

The goal is to learn a reward function  
$$
r_\phi(x, y)
$$  
that assigns a scalar value to a model output $y$ given an input $x$.  

Here, $x \sim \mathcal{D}$ represents a sample drawn from the data distribution,  and $ y \sim \pi_\theta(y \mid x) $ is a response generated by the language model.
This value represents how *preferred* or *relevant* the output is, acting as a proxy for human feedback.


### Approach

A common method is to frame reward modeling as a **regression task**, where the model predicts an **unbounded scalar reward**. 
 
We use a pretrained language model $\pi_\theta(y \mid x)$ and attach a final linear layer with a single neuron, which will be trained to output the predicted reward value $r_\phi(x, y)$ from the positive and negative prompts in the prompt database $\mathcal{D}$.


### Intuition

The reward model learns to **score** responses so that higher rewards correspond to outputs that align better with human preferences.  
This model will later guide reinforcement learning updates (e.g., in PPO) to fine-tune the policy model.

In [ ]:
import transformers
from transformers import AutoModelForSequenceClassification
from trl import RewardConfig, RewardTrainer
from peft import LoraConfig 
import pandas as pd 
import torch
import datasets
from huggingface_hub import login

### Training the Reward Model

The reward model $r_\phi(x, y)$ is trained to predict human (or synthetic) preferences over pairs of model outputs.

---

#### Preference-Based Objective

Given a prompt $x$ and two candidate responses $(y^+, y^-)$,  
where $y^+$ is preferred over $y^-$ according to human feedback,  
the model should assign a higher reward to $y^+$:

$$
r_\phi(x, y^+) > r_\phi(x, y^-)
$$

To enforce this, we use a **pairwise logistic loss** (used in RLHF, e.g. in InstructGPT):

$$
\mathcal{L}_{\text{RM}}(\phi)
= - \mathbb{E}_{(x, y^+, y^-) \sim \mathcal{D}}
  \left[
    \log \sigma\!\left(r_\phi(x, y^+) - r_\phi(x, y^-)\right)
  \right]
$$

where $\sigma(\cdot)$ is the sigmoid function:
$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

This encourages the model to assign a higher score to the preferred completion.

---

#### Intuition

- If $r_\phi(x, y^+) \gg r_\phi(x, y^-)$,  
  then $\sigma(r_\phi(x, y^+) - r_\phi(x, y^-)) \approx 1$,  
  and the loss is small.  
- If the model ranks them incorrectly, the loss is large.  

---

#### Alternative (Regression) Objective

If explicit preference pairs are unavailable,  
the reward model can also be trained via regression to approximate scalar feedback values:

$$
\mathcal{L}_{\text{reg}}(\phi)
= \mathbb{E}_{(x, y, R) \sim \mathcal{D}}
  \left[ (r_\phi(x, y) - R)^2 \right]
$$

---

We will use pairwise as the training objective rather than the regression training loss.



In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct", dtype=torch.bfloat16)
dataset = datasets.load_dataset("trl-lib/ultrafeedback_binarized", split="train")
# Configuration for the Reward Model training loop
reward_config = RewardConfig(bf16=False)
# important to include the score head when base model is not a sequence classification model
lora_config = LoraConfig(modules_to_save=["score"])

reward_trainer = RewardTrainer(
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    train_dataset=dataset,
    peft_config=lora_config,
)

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at HuggingFaceTB/SmolLM2-135M-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
reward_trainer.train()